In [1]:
import pprint
from typing import Any, Dict
import pandas as pd
from langchain_classic.output_parsers import PandasDataFrameOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from langchain_openai import ChatOpenAI
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()
logging.langsmith("CH03-Pandas-DF-OutputParser")

LangSmith 추적을 시작합니다.
[프로젝트명]
CH03-Pandas-DF-OutputParser


In [2]:
model = ChatOpenAI(temperature=0, model="gpt-4o-mini")

In [3]:
def format_parser_output(parser_output: Dict[str, Any]) -> None:
    for key in parser_output.keys():
        parser_output[key] = parser_output[key].to_dict()
    return pprint.PrettyPrinter(width=4, compact=True).pprint(parser_output)

In [4]:
df = pd.read_csv("./data/titanic.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [5]:
parser = PandasDataFrameOutputParser(dataframe=df)
print(parser.get_format_instructions())


The output should be formatted as a string as the operation, followed by a colon, followed by the column or row to be queried on, followed by optional array parameters.
1. The column names are limited to the possible columns below.
2. Arrays must either be a comma-separated list of numbers formatted as [1,3,5], or it must be in range of numbers formatted as [0..4].
3. Remember that arrays are optional and not necessarily required.
4. If the column is not in the possible columns or the operation is not a valid Pandas DataFrame operation, return why it is invalid as a sentence starting with either "Invalid column" or "Invalid operation".

As an example, for the formats:
1. String "column:num_legs" is a well-formatted instance which gets the column num_legs, where num_legs is a possible column.
2. String "row:1" is a well-formatted instance which gets row 1.
3. String "column:num_legs[1,2]" is a well-formatted instance which gets the column num_legs for rows 1 and 2, where num_legs is a p

**gpt-4o-mini**의 경우 출력 값에 "" 이붙어서 출력이 안맞아 짐. 

In [15]:
df_query = "Age Column을 조회해 주세요."

prompt = PromptTemplate(
    template="Answer the user query.\n{format_instructions}\n{question}\n",
    input_variables=["question"],
    partial_variables={
        "format_instructions": parser.get_format_instructions()
    },
)


def clean_output(text: str) -> str:
    # gpt-4o-mini가 가끔 "column:Age" 처럼 앞뒤에 따옴표를 붙여서
    # PandasDataFrameOutputParser가 "column을 요청 타입으로 인식하지 못하는 문제 방지
    return text.strip().strip('"').strip("'")

chain = prompt | model | StrOutputParser() | RunnableLambda(clean_output) | parser
chain_2_comp = prompt | model | StrOutputParser() | RunnableLambda(clean_output)
chain_2_comp_2 = prompt | model | StrOutputParser() 
print("-"*20)
print(chain_2_comp.invoke({"question": df_query}))
print("-"*20)
print(chain_2_comp_2.invoke({"question": df_query}))

# gpt-4o-mini에서는 Error 발생
# chain = prompt | model | parser 
parser_output = chain.invoke({"question": df_query})

# format_parser_output(parser_output)


--------------------
column:Age
--------------------
"column:Age"


In [16]:
format_parser_output(parser_output)

{'Age': {0: 22.0,
         1: 38.0,
         2: 26.0,
         3: 35.0,
         4: 35.0,
         5: nan,
         6: 54.0,
         7: 2.0,
         8: 27.0,
         9: 14.0,
         10: 4.0,
         11: 58.0,
         12: 20.0,
         13: 39.0,
         14: 14.0,
         15: 55.0,
         16: 2.0,
         17: nan,
         18: 31.0,
         19: nan}}


In [ ]:
df_query = "Retrieve the first row." #첫번째 행을 조회
parser_output = chain.invoke({"question":df_query})
format_parser_output(parser_output)

{'0': {'Age': 22.0,
       'Cabin': nan,
       'Embarked': 'S',
       'Fare': 7.25,
       'Name': 'Braund, '
               'Mr. '
               'Owen '
               'Harris',
       'Parch': 0,
       'PassengerId': 1,
       'Pclass': 3,
       'Sex': 'male',
       'SibSp': 1,
       'Survived': 0,
       'Ticket': 'A/5 '
                 '21171'}}


In [18]:
df["Age"].head().mean()

np.float64(31.2)

In [19]:
df_query = "Rerieve the average of the Ages from row 0 to 4."
parser_output = chain.invoke({"question": df_query})
print(parser_output)

{'mean': np.float64(31.2)}


In [ ]:
df_query = "Calculate average 'Fare' rate." #통행 요금 전체 평균 
parser_output = chain.invoke({"question": df_query})
print(parser_output)

{'mean': np.float64(22.19937)}


In [ ]:
df["Fare"].mean()